# Cuando el 90% de accuracy engaña

Compararemos un modelo que siempre predice la mayoría, un kNN con muchos vecinos y un Random Forest balanceado. El objetivo es detectar la clase 1, sin ocultar las falsas alarmas. Aquí “vago” describe el comportamiento de las predicciones; técnicamente kNN también se conoce como *lazy learner* porque conserva los ejemplos para predecir.

Este dataset es sintético: su generador incluye ruido y signos contrarios a la descripción de algunos factores de riesgo. Este ejercicio no valida un criterio clínico.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'datasets').is_dir())
df = pd.read_csv(root / 'datasets/alertaNinosMenores24meses(Kaggle)/alertas_ninos_menores_2anos_sintetico.csv')
print(df['alerta_desnutricion'].value_counts(normalize=True).rename('Proporción'))
X = df.drop(columns='alerta_desnutricion')
y = df['alerta_desnutricion']

## Separar antes de aprender

60% entrenamiento, 20% validación y 20% prueba, conservando la proporción de clases. Los imputadores y el escalador aprenden solo del entrenamiento. Validación elige el umbral; prueba se usa al final.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline

X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=42)

preprocessor = ColumnTransformer([
    ('numeric', make_pipeline(SimpleImputer(strategy='mean'), StandardScaler()),
     make_column_selector(dtype_include='number')),
    ('categorical', make_pipeline(SimpleImputer(strategy='most_frequent'),
                                 OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
     make_column_selector(dtype_exclude='number'))
])
X_train_p = preprocessor.fit_transform(X_train)
X_val_p = preprocessor.transform(X_val)
X_test_p = preprocessor.transform(X_test)

## 1. Mayoría y kNN “vago”

El modelo de referencia siempre predice 0. Con `n_neighbors=101`, kNN suaviza las decisiones y puede votar casi siempre por la mayoría. Esto se comprueba con sus predicciones, no se presupone. Puedes cambiar ese valor para experimentar.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier

baseline = DummyClassifier(strategy='most_frequent').fit(X_train_p, y_train)
knn = KNeighborsClassifier(n_neighbors=101, n_jobs=-1).fit(X_train_p, y_train)

## 2. Modelo más complejo y atención a la minoría

Random Forest puede aprender relaciones no lineales. `class_weight='balanced'` da más peso a los errores en la clase minoritaria. Limitamos la profundidad y el tamaño mínimo de hoja para reducir el sobreajuste al ruido.

Compararemos el mismo bosque con umbral 0.5 y con un umbral elegido en validación. Así distinguimos el efecto del umbral del cambio de modelo.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=150, max_depth=10,
                                min_samples_leaf=30, class_weight='balanced',
                                random_state=42, n_jobs=-1)
forest.fit(X_train_p, y_train)

## 3. Elegir el umbral sin mirar la prueba

Maximizamos F2 en validación: da más importancia al recall que a la precisión. Bajar el umbral puede detectar más positivos, pero también producir más falsas alarmas. Es una elección didáctica, no un umbral clínico.

In [ ]:
from sklearn.metrics import precision_recall_curve

p_val = forest.predict_proba(X_val_p)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, p_val)
f2 = 5 * precision[:-1] * recall[:-1] / np.maximum(4 * precision[:-1] + recall[:-1], 1e-12)
threshold = thresholds[np.argmax(f2)]
print(f'Umbral elegido en validación: {threshold:.3f}')

## 4. Comparar sobre la misma prueba

- **Accuracy:** aciertos totales; puede ocultar el fracaso en la minoría.
- **Recall 1:** proporción de alertas reales detectadas.
- **Precision 1:** proporción de alertas emitidas que son correctas.
- **F2:** combina precisión y recall, priorizando recall.
- **AP:** calidad del ranking de probabilidades; la referencia sin discriminación es aproximadamente la prevalencia (10%).

Las matrices muestran los conteos: fila = clase real, columna = predicción. Un buen resultado exige revisar tanto las alertas omitidas como las falsas alarmas.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             fbeta_score, average_precision_score, ConfusionMatrixDisplay)

p_knn = knn.predict_proba(X_test_p)[:, 1]
p_forest = forest.predict_proba(X_test_p)[:, 1]
results = {
    'Siempre 0': (baseline.predict(X_test_p), baseline.predict_proba(X_test_p)[:, 1]),
    'kNN (k=101)': ((p_knn >= 0.5).astype(int), p_knn),
    'RF balanceado (0.5)': ((p_forest >= 0.5).astype(int), p_forest),
    'RF + umbral F2': ((p_forest >= threshold).astype(int), p_forest)
}
rows = []
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, (name, (pred, prob)) in zip(axes, results.items()):
    rows.append({'Modelo': name, 'Accuracy': accuracy_score(y_test, pred),
                 'Recall 1': recall_score(y_test, pred, zero_division=0),
                 'Precision 1': precision_score(y_test, pred, zero_division=0),
                 'F2': fbeta_score(y_test, pred, beta=2, zero_division=0),
                 'AP': average_precision_score(y_test, prob),
                 'Predicciones de clase 1': int(pred.sum())})
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=[0, 1], ax=ax,
                                           cmap='Blues', colorbar=False)
    ax.set_title(name)
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Clase real')
plt.tight_layout()
plt.show()
comparison = pd.DataFrame(rows).set_index('Modelo')
print(comparison.round(3).to_string())

## Interpretación

¿El kNN obtuvo cerca de 90% de accuracy prediciendo casi todo como 0? Mira su recall y la columna de predicciones positivas.

¿El bosque mejora la detección? Compara recall, precisión, F2 y AP. El cambio de umbral no mejora el ranking (AP es igual para ambos resultados del bosque); cambia la decisión y el equilibrio entre omisiones y falsas alarmas.

Si el recall mejora a costa de muchas falsas alarmas, el problema no está “resuelto”. Los siguientes pasos son revisar la generación de etiquetas, la calidad de las variables y el costo de cada error. No elijas más parámetros usando estos resultados de prueba; usa validación o validación cruzada.